# SQLite test

**What it does**
Creates a small test database file, `storage/sqlite_test.sqlite`, with one table `test_records`, writes three records, reads them back, counts them by status, and shows them as a table.

**Why**
To prove SQLite works on this machine (in ING: on the workbench) before the app is switched to `KCT_DB_DRIVER=sqlite`: that Python has the built-in `sqlite3` module and that the project folder is writable. It uses its own file and table, so it never touches the app's data.

**How to use**
1. Open this notebook from the project's `notebooks/` folder.
2. Run all cells (Run > Run All Cells).
3. You should see the SQLite version, the file path, three new records and a count per status. Each run adds three more records.
4. To start over, uncomment the line in the last cell and run it; it deletes the test file.

**Needs**: only Python. pandas is optional, for the last table. Terminal equivalent: `./kct.sh sqlite`.

## Setup

Prints the SQLite version built into this Python and where the test file goes.

In [ ]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

# Test file next to the app's storage; delete it any time.
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = PROJECT / "storage" / "sqlite_test.sqlite"
DB.parent.mkdir(parents=True, exist_ok=True)

print("SQLite version:", sqlite3.sqlite_version)
print("Database file: ", DB)

## Create a table

In [ ]:
with sqlite3.connect(DB) as con:
    # An older 01_setup made a test_records table without a status column: start that one fresh.
    columns = {row[1] for row in con.execute("PRAGMA table_info(test_records)")}
    if columns and "status" not in columns:
        con.execute("DROP TABLE test_records")
    con.execute("""
        CREATE TABLE IF NOT EXISTS test_records (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            name        TEXT NOT NULL,
            status      TEXT NOT NULL,
            created_at  TEXT NOT NULL
        )
    """)
print("Table test_records ready")

## Insert some records

In [ ]:
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
records = [
    ("Verification precedes the reset", "pending", now),
    ("Reset done by the verifying agent", "effective", now),
    ("Ticket references the request", "not_effective", now),
]

with sqlite3.connect(DB) as con:
    con.executemany("INSERT INTO test_records (name, status, created_at) VALUES (?, ?, ?)", records)
print(f"Inserted {len(records)} records")

## Show the records

In [ ]:
with sqlite3.connect(DB) as con:
    rows = con.execute("SELECT id, name, status, created_at FROM test_records ORDER BY id").fetchall()

print(f"{'id':>3}  {'name':<36}  {'status':<14}  created_at")
print("-" * 80)
for row in rows:
    print(f"{row[0]:>3}  {row[1]:<36}  {row[2]:<14}  {row[3]}")

## Count by status

In [ ]:
with sqlite3.connect(DB) as con:
    for status, count in con.execute("SELECT status, COUNT(*) FROM test_records GROUP BY status ORDER BY status"):
        print(f"{status:<14} {count}")

## As a table (only if pandas is installed)

In [ ]:
try:
    import pandas as pd
except ImportError:
    print("pandas not installed; the printed list above is enough")
else:
    with sqlite3.connect(DB) as con:
        display(pd.read_sql_query("SELECT * FROM test_records ORDER BY id", con))

## Clean up (optional)

Deletes the test file.

In [ ]:
# DB.unlink(missing_ok=True); print("Deleted", DB)